# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [4]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [11]:
# Initialize and constants

# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#    print("API key looks good so far")
# else:
#    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
# openai = OpenAI()

MODEL = 'gemma3:4b'
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result) # return JSON string into Python dictionary
    return links
    

In [13]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'about page', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'course', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'course', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'event',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'event',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'event',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'profile', 'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [14]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [15]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemma3:4b
Found 9 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/'},
  {'type': 'event',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'event',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'personal website', 'url': 'https://edwarddonner.com/'},
  {'type': 'personal website',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [16]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma3:4b
Found 14 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'models', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces', 'url': 'https://huggingface.co/spaces'},
  {'type': 'docs', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status updates', 'url': 'https://status.huggingface.co/'},
  {'type': 'github repository', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemma3:4b
Found 26 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5
Updated
36 minutes ago
•
13.9k
•
930
openbmb/MiniCPM-SALA
Updated
2 days ago
•
1.52k
•
392
moonshotai/Kimi-K2.5
Updated
8 days ago
•
686k
•
2.11k
Qwen/Qwen3-Coder-Next
Updated
10 days ago
•
217k
•
820
zai-org/GLM-OCR
Updated
4 days ago
•
693k
•
1.02k
Browse 2M+ models
Spaces
Running
on
A100
Featured
350
ACE-Step v1.5
🎵
350
Music Generation Foundation Model v1.5
Running
613
Demo Playground
⚡
613
Free platform to access multiple AI models
Running
Reachy
151
Reachy Phone Home
📱
151
Phone focus companion for Reachy Mini
Runt

In [19]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma3:4b
Found 26 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-5\nUpdated\nabout 1 hour ago\n•\n13.9k\n•\n930\nopenbmb/MiniCPM-SALA\nUpdated\n2 days ago\n•\n1.52k\n•\n392\nmoonshotai/Kimi-K2.5\nUpdated\n8 days ago\n•\n686k\n•\n2.11k\nQwen/Qwen3-Coder-Next\nUpdated\n10 days ago\n•\n217k\n•\n820\nzai-org/GLM-OCR\nUpdated\n4 days ago\n•\n693k\n•\n1.02k\nBrowse 2M+ models\nSpaces\nRunning\non\nA100\nFeatured\n350\nACE-Step v1.5\n🎵\n350\nMusic G

In [25]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma3:4b
Found 20 relevant links


## Hugging Face: The AI Community Building the Future

**(Brochure for Prospective Customers, Investors, and Recruiters)**

**Introduction:**

Hugging Face is the world’s leading AI community, empowering developers and researchers to build, discover, and collaborate on cutting-edge machine learning models, datasets, and applications. We’re fundamentally changing how AI is developed and deployed.

**What We Offer:**

* **A Vast Ecosystem:**  Access to an unparalleled collection of over 2 million models, 500,000+ datasets, and 1 million+ applications. 
* **Collaboration Platform:** Our platform fosters a thriving community where users can easily share, experiment with, and improve AI models.
* **Diverse Modalities:**  We support a wide range of data modalities including text, image, video, audio, and even 3D, enabling truly versatile AI solutions.


**Key Features & Offerings:**

* **Models:** Explore a curated selection of trending models like zai-org/GLM-5, Qwen/Qwen3-Coder-Next and many others, constantly updated with the latest advancements.
* **Spaces:**  Quickly deploy and showcase your AI applications with Spaces – a free platform for running AI models and building interactive demos. See featured Spaces like ACE-Step v1.5, Qwen3-TTS Demo, and Wan2.2 Animate.
* **Datasets:** Discover massive datasets supporting a diverse range of applications, including openbmb/UltraData-Math andtencent/CL-bench.
* **Enterprise Solutions:** Accelerate your AI initiatives with our Team & Enterprise solutions, offering dedicated support, enterprise-grade security, and access controls.



**Our Community & Culture:**

At Hugging Face, we believe in open source, collaboration, and continuous learning.  We foster a:

* **Collaborative Environment:**  Our community is driven by a shared passion for advancing the state of AI.
* **Open Source Focused:**  We strongly support open source principles and contribute significantly to the open-source AI ecosystem.
* **Innovative Culture:**  We encourage experimentation, rapid iteration, and pushing the boundaries of what’s possible with AI.



**For Customers:**

* **Accelerated AI Development:**  Our extensive model library and collaborative tools significantly reduce the time and cost of building AI applications.
* **Fast Innovation:** Jumpstart your development with our pre-trained models and quickly customize them for your specific needs.
* **Scalable Solutions:** Utilize our dedicated compute options to ensure your models can handle the load.



**For Investors:**

* **Massive Market Opportunity:** The AI market is experiencing exponential growth, and Hugging Face is at the forefront of this revolution.
* **Strong Community Traction:**  Our vibrant and engaged community drives adoption and innovation.
* **Strategic Partnerships:** We are actively building strategic partnerships with leading technology companies and research institutions.



**For Recruiters:**

* **Join a Dynamic Team:**  Hugging Face is a fast-paced, intellectually stimulating environment where you can make a real impact.
* **Cutting-Edge Technology:** Work with the latest AI technologies and contribute to groundbreaking projects.
* **Growth Opportunities:**  We are a rapidly growing company, offering significant career advancement opportunities. 

**Careers at Hugging Face:**  [Link to Careers Page - *Not provided in content*]

**Start Building Today:**  Sign up for free at [https://huggingface.co/](https://huggingface.co/)  
  
**Follow Us:** [Links to social media - *Not provided in content*]

---

**Note:** This brochure was created based solely on the provided text.  A more comprehensive brochure could benefit from including visual elements and details from the full Hugging Face website.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

# GitHub Copilot

Looking at your code, the `KeyError "links"` occurs in the `fetch_page_and_all_relevant_links()` function at line 7 where it tries to access `relevant_links['links']`.

The issue is that `select_relevant_links()` is calling the Ollama model with `response_format={"type": "json_object"}`, but **Ollama's Gemma model may not properly support structured JSON output**, causing it to return invalid JSON or a response without the expected "links" key.

Here's the fix:



In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ]
        # Removed response_format for Ollama compatibility
    )
    result = response.choices[0].message.content
    try:
        links = json.loads(result)
        # Ensure the expected structure exists
        if "links" not in links:
            print("Warning: 'links' key not found in response")
            links = {"links": []}
    except json.JSONDecodeError:
        print(f"Failed to parse JSON response: {result}")
        links = {"links": []}
    
    print(f"Found {len(links['links'])} relevant links")
    return links



**Changes made:**
1. Removed `response_format={"type": "json_object"}` (Ollama doesn't support this well)
2. Added error handling for JSON parsing failures
3. Added a fallback structure if the "links" key is missing

Try this and the KeyError should be resolved!

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>